GPT - Generatively pre-trained transformer

We are going to use Shakespear text to train transformers.

In [2]:
# We always start with a dataset to train on. Let's download the tiny shakespeare dataset
!wget https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt

zsh:1: command not found: wget


In [3]:
import urllib.request

url = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"
urllib.request.urlretrieve(url, "input.txt")

('input.txt', <http.client.HTTPMessage at 0x1073490c0>)

In [4]:
# read it in to inspect it
with open('input.txt', 'r', encoding='utf-8') as f:
    text = f.read()

In [5]:
print("length of dataset in characters: ", len(text))

length of dataset in characters:  1115394


In [6]:
# let's look at the first 1000 characters
print(text[:1000])

First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You are all resolved rather to die than to famish?

All:
Resolved. resolved.

First Citizen:
First, you know Caius Marcius is chief enemy to the people.

All:
We know't, we know't.

First Citizen:
Let us kill him, and we'll have corn at our own price.
Is't a verdict?

All:
No more talking on't; let it be done: away, away!

Second Citizen:
One word, good citizens.

First Citizen:
We are accounted poor citizens, the patricians good.
What authority surfeits on would relieve us: if they
would yield us but the superfluity, while it were
wholesome, we might guess they relieved us humanely;
but they think we are too dear: the leanness that
afflicts us, the object of our misery, is as an
inventory to particularise their abundance; our
sufferance is a gain to them Let us revenge this with
our pikes, ere we become rakes: for the gods know I
speak this in hunger for bread, not in thirst for revenge.



In [7]:
# here are all the unique characters that occur in this text
chars = sorted(list(set(text)))
vocab_size = len(chars)
print(''.join(chars))
print(vocab_size)


 !$&',-.3:;?ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz
65


# Tokenization
Below we do  simplest tokenization - character level tokenization
Google uses Sentencepiece - https://github.com/google/sentencepiece
OpenAi user tiktoken that uses BPE - https://github.com/openai/tiktoken

# Original BPE
compression or encodign  technique to encode longer strigns into shorter string using a translation table
Encode most common continuous sequesnce characters in a string with unsued 'placeholder' bytes. The iteration ends when no sequences can be found, leaving the target text effectively compressed

Example
aaabdaaabac
The byte pair "aa" occurs most often, so it will be replaced by a byte that is not used in the data, such as "Z"
```ZabdZabac
Z=aa```
Then the process is repeated with byte pair "ab", replacing it with "Y":
```
ZYdZYac
Y=ab
Z=aa
```

## Modified BPE
odified BPE does not aim to maximally compress text, but rather, to encode plaintext into "tokens", which are natural numbers
- All the unique tokens found in a corpus are listed in a token vocabulary. The token vocabulary can also include some other special tokens, relative to the use case
- vocabulary, in the case of GPT-3.5 and GPT-4, is 100258 (100000 from BPE algorithm and 258 included as special tokens)

Example:
Suppose we are encoding the previous example of "aaabdaaabac", with a specified vocabulary size of 6, then it would first be encoded as "0, 0, 0, 1, 2, 0, 0, 0, 1, 0, 3" with a vocabulary of "a=0, b=1, d=2, c=3". Then it would proceed as before, and obtain "4, 5, 2, 4, 5, 0, 3" with a vocabulary of "a=0, b=1, d=2, c=3, aa=4, ab=5".

So far this is essentially the same as before. However, if we only had specified a vocabulary size of 5, then the process would stop at vocabulary "a=0, b=1, d=2, c=3, aa=4", so that the example would be encoded as "4, 0, 1, 2, 4, 0, 1, 0, 3". Conversely, if we had specified a vocabulary size of 8, then it would be encoded as "7, 6, 0, 3", with a vocabulary of "a=0, b=1, d=2, c=3, aa=4, ab=5, aaab=6, aaabd=7". This is not maximally compressed, because modified BPE does not aim for maximum compression. Instead, it aims for an encoding that is efficient and practical for language model training


In [8]:
# create a mapping from characters to integers
stoi = { ch:i for i,ch in enumerate(chars) }
itos = { i:ch for i,ch in enumerate(chars) }
encode = lambda s: [stoi[c] for c in s] # encoder: take a string, output a list of integers
decode = lambda l: ''.join([itos[i] for i in l]) # decoder: take a list of integers, output a string

print(encode("hii there"))
print(decode(encode("hii there")))

[46, 47, 47, 1, 58, 46, 43, 56, 43]
hii there


In [ ]:
# let's now encode the entire text dataset and store it into a torch.Tensor
import torch # we use PyTorch: https://pytorch.org
data = torch.tensor(encode(text), dtype=torch.long)
print(data.shape, data.dtype)
print(data[:1000]) # the 1000 characters we looked at earier will to the GPT look like this

In [10]:
# Let's now split up the data into train and validation sets
n = int(0.9*len(data)) # first 90% will be train, rest val
train_data = data[:n]
val_data = data[n:]

we don't train on entire dataset but only on chunks, why?

Why We Train on Chunks Instead of the Entire Dataset
There are 3 main reasons:
1. GPU Memory Constraints
The Shakespeare dataset has 1,115,394 characters
Training on the full sequence at once would require computing attention between ALL tokens simultaneously
Transformer attention has O(n²) memory complexity — for 1M tokens, that's ~1 trillion attention computations
This would exceed GPU memory limits instantly
2. Computational Efficiency
By using a fixed block_size (e.g., 8 or 256), we process manageable chunks
Each chunk becomes a training example with a predictable memory footprint
We can batch multiple chunks together for parallel processing
3. Better Learning Signal (Multiple Training Examples)
From a single sequence of length n, we get many overlapping training examples
With `block_size = 8`, the sequence `[18, 47, 56, 57, 58, 1, 15, 47, 58]` gives us:
```python
#Context → Target[18]                    → 47[18, 47]                → 56  [18, 47, 56]            → 57[18, 47, 56, 57]        → 58[18, 47, 56, 57, 58]    → 1# ... and so on
```
This trains the model to predict the next token given contexts of varying lengths (1 to block_size), making the model robust at inference time when it needs to handle different context sizes.

**In Summary**
Training on chunks is a practical necessity (memory/compute) that also provides a pedagogical benefit (varied context lengths for better generalization).

In [ ]:
block_size = 8
train_data[:block_size+1]


tensor([18, 47, 56, 57, 58,  1, 15, 47, 58])

# Why block_size + 1?
When you see `train_data[:block_size+1]`, the +1 is needed because we need both inputs (x) and targets (y) from the same chunk.

Index:    0    1    2    3    4    5    6    7    8
Data:    [18,  47,  56,  57,  58,   1,  15,  47,  58]
          ↑_________________________↑    ↑
              x (inputs, 0-7)           y (last target)

In [12]:

x = train_data[:block_size]
y = train_data[1:block_size+1]
for t in range(block_size):
    context = x[:t+1]
    target = y[t]
    print(f"when input is {context} the target is {target}")
    


when input is tensor([18]) the target is 47
when input is tensor([18, 47]) the target is 56
when input is tensor([18, 47, 56]) the target is 57
when input is tensor([18, 47, 56, 57]) the target is 58
when input is tensor([18, 47, 56, 57, 58]) the target is 1
when input is tensor([18, 47, 56, 57, 58,  1]) the target is 15
when input is tensor([18, 47, 56, 57, 58,  1, 15]) the target is 47
when input is tensor([18, 47, 56, 57, 58,  1, 15, 47]) the target is 58


In [ ]:
# above is time dimension of the dataset 
# we intorduce batch dimention fo the dataset so that batches are run in parallel on the gpu
torch.manual_seed(1337)
batch_size = 4 # how many independent sequences will we process in parallel?
block_size = 8 # what is the maximum context length for predictions?

def get_batch(split):
    data = train_data if split == 'train' else val_data
    ix = torch.randint(len(data) - block_size, (batch_size,)) # batch_size number of random offsets 
    # ix is going to be a tensor of shape (batch_size,)
    # it's going to contain random integers between 0 and len(data) - block_size
    # these integers will be the starting indices of the chunks that we're going to take from the data
    x = torch.stack([data[i:i+block_size] for i in ix]) # (B, T)
    y = torch.stack([data[i+1:i+block_size+1] for i in ix]) # (B, T)
    return x, y

xb, yb = get_batch('train')
print('inputs:')
print(xb.shape)
print(xb)
print('targets:')
print(yb.shape)
print(yb)

print('--------------------------------')
for b in range(batch_size):
    for t in range(block_size):
        context = xb[b, :t+1]
        target = yb[b, t]
        print(f"when input is {context} the target is {target}")
        
# 32 exampels are compltely random for transformers packed into a batch

inputs:
torch.Size([4, 8])
tensor([[24, 43, 58,  5, 57,  1, 46, 43],
        [44, 53, 56,  1, 58, 46, 39, 58],
        [52, 58,  1, 58, 46, 39, 58,  1],
        [25, 17, 27, 10,  0, 21,  1, 54]])
targets:
torch.Size([4, 8])
tensor([[43, 58,  5, 57,  1, 46, 43, 39],
        [53, 56,  1, 58, 46, 39, 58,  1],
        [58,  1, 58, 46, 39, 58,  1, 46],
        [17, 27, 10,  0, 21,  1, 54, 39]])
--------------------------------
when input is tensor([24]) the target is 43
when input is tensor([24, 43]) the target is 58
when input is tensor([24, 43, 58]) the target is 5
when input is tensor([24, 43, 58,  5]) the target is 57
when input is tensor([24, 43, 58,  5, 57]) the target is 1
when input is tensor([24, 43, 58,  5, 57,  1]) the target is 46
when input is tensor([24, 43, 58,  5, 57,  1, 46]) the target is 43
when input is tensor([24, 43, 58,  5, 57,  1, 46, 43]) the target is 39
when input is tensor([44]) the target is 53
when input is tensor([44, 53]) the target is 56
when input is tensor

## Bigram Language Model - Detailed Explanation

### What is a Bigram Model?

A **bigram model** is the simplest possible language model. It predicts the next token based **only on the current token** — no history, no context. The question it answers is: "Given token X, what's likely to come next?"

---

### Understanding the Embedding Table

```python
self.token_embedding_table = nn.Embedding(vocab_size, vocab_size)
```

This creates a **learnable lookup table** of shape `(65, 65)` since our `vocab_size = 65` (unique characters in Shakespeare).

| Dimension | Meaning |
|-----------|---------|
| **Rows (65)** | One row for each possible input token (0-64) |
| **Columns (65)** | Logits (unnormalized scores) for each possible next token |

**Think of it as:** A 65×65 matrix where row `i` contains the model's predictions for "what comes after token `i`"

---

### The Forward Pass

```python
def forward(self, idx, targets):
    logits = self.token_embedding_table(idx)  # Output: (B, T, C)
```

**Input shapes:**
- `idx` = input tokens with shape `(Batch, Time)` e.g., `(4, 8)` means 4 sequences of 8 tokens each
- `targets` = target tokens (not used in this forward pass yet)

**What happens internally:**

For each token in `idx`, we "pluck out" that token's row from the embedding table:

```
idx = [[24, 43, 58, ...], ...]

Token 24 → Look up row 24 → Get 65 logits for "what comes after 24"
Token 43 → Look up row 43 → Get 65 logits for "what comes after 43"
Token 58 → Look up row 58 → Get 65 logits for "what comes after 58"
...
```

**Output:** `logits` with shape `(B, T, C)` = `(4, 8, 65)`
- B=4 batches
- T=8 time steps
- C=65 logit scores per position

---

### The Critical Limitation: No Context!

Each token makes its prediction **completely independently**:

```
Token "h" → predicts next (doesn't know it's in "hello")
Token "e" → predicts next (doesn't know "h" came before)
Token "l" → predicts next (doesn't know "he" came before)
```

**The tokens are NOT talking to each other.** They only see themselves — that's why this is called a "bigram" model (it only considers pairs: current → next).

---

### What are Logits?

Logits are **unnormalized log-probabilities**. To convert them to actual probabilities:

```python
probs = F.softmax(logits, dim=-1)  # Each row now sums to 1.0
```

---

### Why Start with Bigram?

This simple model is a stepping stone. Later, we'll add **self-attention** which allows tokens to "communicate" with each other and use context for much better predictions.


In [21]:
# now let's feed that into a simple bigram model
import torch
import torch.nn as nn
from torch.nn import functional as F
torch.manual_seed(1337)

class BigramLanguageModel(nn.Module):
    def __init__(self, vocab_size):
        # here we are creating a token embedding table of size vocab_size x vocab_size
        super().__init__()
        # each token directly reads off the logits for the next token from a lookup table
        self.token_embedding_table = nn.Embedding(vocab_size, vocab_size)
    
    def forward(self, idx, targets=None):
        # idx and targets are both (B, T) tensor of integers
        # pluck out the row of embedding of the index
        # e.g. [24] it will pluck out 24th row of the embedding table
        
        logits = self.token_embedding_table(idx) # (Batch, Time/Sequence, Channel)
        # and so what's happening here is we are predicting what comes next based on just the individual identity of a single
        # token in the sequence, currently the tokens are not talking to each other and they're not
        # seeing any context except for they're just seeing themselves so I'm a f I'm a token number 24 and then I can
        # see what the logits are for the next token so the logits are the unnormalized probabilities of the next token

        # negative log likelihood loss
        # loss = F.cross_entropy(logits, targets) # this will not work because the logits are multi-dimensional
        # by default pytorch cross_entropy function want C to be the last dimension of the tensor
        # our logist are multi-dimensional wso we ned to convert to 2-dmensional to use cross entropy
        if targets is None:
            loss = None
        else:
            B,T,C = logits.shape
            logits = logits.view(B*T, C) # B*T -> stretch the logit index to 1-d sequence and preseve the channel
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)
        return logits, loss

    # we need to generate a new token based on the context of the previous tokens
    def generate(self, idx, max_new_tokens):
        # this fucntion takes (B, T) array of indices in the current context in a batch and generate (B, T+1, +2, ....)
        # idx is (B, T) array of indices in the current context in a batch
        for _ in range(max_new_tokens):
            # get the predictions
            logits, loss = self(idx) # (B, T, C)
            # focus only on the last time step, because thtese are the logits for the next token
            logits = logits[:, -1, :] # (B, C)
            # apply softmax to get probabilities
            probs = F.softmax(logits, dim=-1)
            # sample from the distribution
            idx_next = torch.multinomial(probs, num_samples=1) # (B, 1)
            # append sampled index to the running sequence
            idx = torch.cat((idx, idx_next), dim=-1) # (B, T+1)
        return idx
            
    
m = BigramLanguageModel(vocab_size)
logits, loss = m(xb, yb)
print(logits.shape)
print(loss) # we are expecting -ln(1/65) = 4.17
# idx = torch.zeros((1, 1), dtype=torch.long)
print(decode(m.generate(torch.zeros((1, 1), dtype=torch.long), max_new_tokens=100)[0].tolist()))





torch.Size([32, 65])
tensor(4.8786, grad_fn=<NllLossBackward0>)

SKIcLT;AcELMoTbvZv C?nq-QE33:CJqkOKH-q;:la!oiywkHjgChzbQ?u!3bLIgwevmyFJGUGp
wnYWmnxKWWev-tDqXErVKLgJ


In [22]:
# train the model
optimizer = torch.optim.AdamW(m.parameters(), lr=1e-3)


In [28]:
batch_size = 32
for steps in range(10000):
    xb, yb = get_batch('train')
    logits, loss = m(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

print(loss.item())
    


2.516516923904419


In [29]:
print(decode(m.generate(torch.zeros((1, 1), dtype=torch.long), max_new_tokens=100)[0].tolist()))



Wherrowintherinche's d t nd alf d'sss:

Bu, we t ssthinon t, anthard wo amy, thir lame not
CAne g go


# Self-attention block

# Self-attention math trick


In [ ]:
torch.manual_seed(1337)
B, T, C = 4, 8, 2 # batch, time, channel
x = torch.randn(B, T, C)

# let's see what the shape of x is
print(x.shape)

In [ ]:


# right now tokens are not talking to each other,
# we like to them to talk to each other
# information should flow from previous token to current token
# e.g. if at 5th token we want the context of 4,3,2,1,

# simplest way to preserve teh prededing token is takign the average of the previous tokens
# ie. we like to take channel at current token and also the channels of previous tokens 
# average them up and then use that feature vector as a context for the current token.
# averga is weak form of context, we need to do better

xbow = torch.zeros((B, T, C))
for b in range(B):
    for t in range(T):
        xprev = x[b, :t+1] # (t, C)
        xbow[b, t, :] = torch.mean(xprev, dim=0)
print(xbow.shape)

# the trick is dooing above in matrix multiplication
# let's look at simple matrix multiplication
torch.manual_seed(42)
a = torch.ones(3, 3) # 2 x 3 matrix
print('a', a)
b = torch.randint(0, 10, (3, 2)).float() # 3 x 2 matrix
print('b', b)
print('a @ b', a @ b)

# verstion 2
# the trick here is to use torch.tril to mask the upper triangular part of the matrix
a = a * torch.tril(torch.ones(3, 3))
print('a', a)
print('a @ b', a @ b)
# now `a @b` will have rows that are sum of previous rows
# by normalising the tril we can get the average
a = torch.tril(torch.ones(3, 3))
a = a / torch.sum(a, dim=1, keepdim=True)
print('a', a)
print('a @ b', a @ b)

# now let's vectorise the double for loop
wei = torch.tril(torch.ones(T, T)) # weights 
wei = wei / wei.sum(dim=1, keepdim=True)
print('wei', wei)
xbow2 = wei @ x # (4, 8, 8) @ (4, 8, 2) -> (4, 8, 2)
print('xbow2', xbow2)
torch.allclose(xbow, xbow2) # they are the same

# above is weighted aggregation 

# version 3
# let's do another version usign softmax
wei = torch.zeros((T, T))
wei = wei.masked_fill(torch.tril(torch.ones(T, T)), float('-inf')) # make the upper triangular part of the matrix -inf
wei = F.softmax(wei, dim=1) # softwmax is also a  normalisation operation that takes exponent and divides by sum
xbow3 = wei @ x 

# wei = torch.zeros((T, T))
# this v3 is better because think of wei above  as like an interaction matrix or affinity 
# which is telling us how much each token from the past do we want to aggregate and averahe io 

# this like `wei = wei.masked_fill(torch.tril(torch.ones(T, T)), float('-inf'))` is saying tokens from the future should not be considered

# Summary 
# short from this entire section is that you can do weighted aggregations of your past
# Elements by having by using matrix multiplication of a lower triangular
# fashion and then the elements here in the lower triangular part are telling you how much of each element uh fuses into this position


torch.Size([4, 8, 2])
a tensor([[1., 1., 1.],
        [1., 1., 1.],
        [1., 1., 1.]])
b tensor([[2., 7.],
        [6., 4.],
        [6., 5.]])
a @ b tensor([[14., 16.],
        [14., 16.],
        [14., 16.]])
a tensor([[1., 0., 0.],
        [1., 1., 0.],
        [1., 1., 1.]])
a @ b tensor([[ 2.,  7.],
        [ 8., 11.],
        [14., 16.]])
a tensor([[1.0000, 0.0000, 0.0000],
        [0.5000, 0.5000, 0.0000],
        [0.3333, 0.3333, 0.3333]])
a @ b tensor([[2.0000, 7.0000],
        [4.0000, 5.5000],
        [4.6667, 5.3333]])
wei tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.5000, 0.5000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.3333, 0.3333, 0.3333, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.2500, 0.2500, 0.2500, 0.2500, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.2000, 0.2000, 0.2000, 0.2000, 0.2000, 0.0000, 0.0000, 0.0000],
        [0.1667, 0.1667, 0.1667, 0.1667, 0.1667, 0.1667, 0.0000, 0.0000],
        [0.

True

## Weighted Aggregation: The Mathematical Foundation of Self-Attention

This section demonstrates a **fundamental building block** of Transformers: how tokens can "communicate" with past tokens using matrix multiplication. This is the precursor to self-attention.

---

### The Problem: Tokens in Isolation

In our Bigram model, each token predicts the next token **without any context**. Token at position 5 doesn't know what happened at positions 1, 2, 3, 4.

**Goal:** We want information to flow from previous tokens to the current token.

---

### Solution: Weighted Aggregation of Past Tokens

The simplest way to give a token "context" is to **average** all the tokens that came before it.

---

## Version 1: Naive For-Loop Implementation

```python
xbow = torch.zeros((B, T, C))
for b in range(B):
    for t in range(T):
        xprev = x[b, :t+1]           # All tokens from 0 to t (inclusive)
        xbow[b, t, :] = torch.mean(xprev, dim=0)  # Average them
```

For each position `t`, we take the mean of all tokens from position 0 to t:

| Position t | Tokens Averaged | Result |
|------------|-----------------|--------|
| 0 | x[0] | x[0] |
| 1 | x[0], x[1] | (x[0] + x[1]) / 2 |
| 2 | x[0], x[1], x[2] | (x[0] + x[1] + x[2]) / 3 |
| ... | ... | ... |

**Problem:** This nested loop is slow! O(B × T × T) operations.

---

## Version 2: Matrix Multiplication with Lower Triangular Matrix

### The Key Insight

Matrix multiplication can compute **weighted sums** in parallel!

Consider a simple example with 3 tokens:

```
a = [[1, 0, 0],      b = [[2, 7],
     [1, 1, 0],           [6, 4],
     [1, 1, 1]]           [6, 5]]
```

When we compute `a @ b`:

```
Row 0: 1×[2,7] + 0×[6,4] + 0×[6,5] = [2, 7]      ← Only token 0
Row 1: 1×[2,7] + 1×[6,4] + 0×[6,5] = [8, 11]    ← Sum of tokens 0,1
Row 2: 1×[2,7] + 1×[6,4] + 1×[6,5] = [14, 16]   ← Sum of tokens 0,1,2
```

### The Lower Triangular Matrix (tril)

```python
torch.tril(torch.ones(3, 3))
```
produces:
```
[[1, 0, 0],
 [1, 1, 0],
 [1, 1, 1]]
```

This **masks out the future** — each row only "sees" current and past positions.

### Normalizing to Get Averages

To get averages instead of sums, divide each row by its sum:

```python
a = torch.tril(torch.ones(3, 3))
a = a / a.sum(dim=1, keepdim=True)
```
produces:
```
[[1.0000, 0.0000, 0.0000],   ← 1/1
 [0.5000, 0.5000, 0.0000],   ← 1/2, 1/2
 [0.3333, 0.3333, 0.3333]]   ← 1/3, 1/3, 1/3
```

Now `a @ b` computes **running averages** in one matrix multiply!

---

## The Mathematical Formulation

For a sequence of T tokens with C channels, the weighted aggregation is:

$$\text{xbow}[t] = \sum_{i=0}^{t} w_{t,i} \cdot x[i]$$

where $w_{t,i}$ are the weights (and $\sum_i w_{t,i} = 1$ for averaging).

In matrix form:
$$\text{xbow} = W \cdot X$$

where $W$ is a lower triangular weight matrix of shape $(T, T)$ and $X$ is $(T, C)$.

---

## Version 3: Using Softmax (The Attention Pattern!)

```python
wei = torch.zeros((T, T))
wei = wei.masked_fill(tril == 0, float('-inf'))  # Mask future positions
wei = F.softmax(wei, dim=-1)                      # Normalize
xbow3 = wei @ x
```

### Why Softmax?

1. **Exponential + Normalize:** `softmax(z)_i = exp(z_i) / Σexp(z_j)`
2. **Masked positions:** `-inf` → `exp(-inf) = 0` → completely ignored
3. **Unmasked positions:** `0` → `exp(0) = 1` → equal weight after normalization

### The Affinity Matrix

The `wei` matrix is an **affinity matrix** — it defines how much each past token contributes to the current token's representation:

```
wei = [[1.00, 0.00, 0.00, 0.00, ...],   ← Token 0 sees only itself
       [0.50, 0.50, 0.00, 0.00, ...],   ← Token 1 sees tokens 0,1 equally
       [0.33, 0.33, 0.33, 0.00, ...],   ← Token 2 sees tokens 0,1,2 equally
       ...]
```

**This is exactly what self-attention does, but with LEARNED weights instead of uniform ones!**

---

## From Uniform Weights to Self-Attention

| Version | Weight Matrix | Description |
|---------|---------------|-------------|
| **V1-V3 (here)** | Fixed uniform weights | Every past token contributes equally |
| **Self-Attention** | Learned data-dependent weights | Tokens learn which past tokens are relevant |

In self-attention, the weights come from:
$$w_{t,i} = \text{softmax}\left(\frac{Q_t \cdot K_i^T}{\sqrt{d_k}}\right)$$

where Q (Query) and K (Key) are learned projections of the tokens.

---

## Why is This Important for Language Models?

### Causal Masking

The lower triangular structure enforces **causality**:
- Token at position `t` can ONLY see tokens at positions `0, 1, ..., t`
- It CANNOT see future tokens `t+1, t+2, ...`
- This is essential for autoregressive language models that predict the next token

### Parallelization

Unlike RNNs that process sequentially, this matrix multiplication approach:
- Computes all positions in parallel
- Leverages GPU matrix operations
- Enables efficient training on long sequences

---

## Visual Summary

```
Input x:     [x₀]  [x₁]  [x₂]  [x₃]  ...
              ↓     ↓     ↓     ↓
              ┌─────┴─────┴─────┴─────┐
              │   Lower Triangular    │
              │    Weight Matrix W    │
              └─────┬─────┬─────┬─────┘
                    ↓     ↓     ↓
Output xbow: [x₀] [avg(x₀,x₁)] [avg(x₀,x₁,x₂)] ...

Each output position is a weighted combination of 
all previous inputs (including itself).
```

---

## Key Takeaways

1. **Matrix multiplication can compute weighted aggregations** of sequences efficiently
2. **Lower triangular matrices** enforce causality (no peeking at the future)
3. **Softmax normalization** ensures weights sum to 1 and handles masking gracefully
4. This is the **foundation of self-attention** — the only difference is that attention learns the weights instead of using uniform averages
5. The weight matrix `wei` will become the **attention scores** in the full Transformer


## Attention Variations Across Language Models

The weighted aggregation mechanism we just learned is the foundation of **attention**. Different language models have evolved various attention patterns to balance performance, efficiency, and context length. Here's a comprehensive overview:

---

## 1. Standard Self-Attention (Transformer - 2017)

**Used in:** Original Transformer, BERT, GPT-1/2

```
Attention(Q, K, V) = softmax(QK^T / √d_k) × V
```

### Masking Patterns

| Model Type | Mask | Can See |
|------------|------|---------|
| **Encoder (BERT)** | Bidirectional (no mask) | All tokens in sequence |
| **Decoder (GPT)** | Causal (lower triangular) | Only past tokens |
| **Encoder-Decoder (T5)** | Cross-attention | Encoder sees all; Decoder is causal |

### Complexity
- **Time:** O(n²) — every token attends to every other token
- **Memory:** O(n²) — must store full attention matrix
- **Limitation:** Doesn't scale well beyond ~2K-4K tokens

---

## 2. Multi-Head Attention

**Used in:** All modern Transformers (GPT, BERT, LLaMA, etc.)

Instead of one attention, run **h parallel attention heads** with smaller dimensions:

```python
# Instead of one 512-dim attention:
head_dim = 512 // 8  # = 64
heads = [Attention(Q_i, K_i, V_i) for i in range(8)]
output = Concat(heads) @ W_o
```

### Why Multiple Heads?

Each head can learn **different attention patterns**:
- Head 1: Focus on syntax (subject-verb agreement)
- Head 2: Focus on nearby tokens (local context)
- Head 3: Focus on semantic relationships
- Head 4: Focus on positional patterns

### Visualization

```
                    ┌─── Head 1: "The [cat] sat" ───┐
                    │                               │
Input ─────────────►├─── Head 2: "[The] cat [sat]" ─┼──► Concat ──► Output
"The cat sat"       │                               │
                    └─── Head 3: "The cat [sat]" ───┘
```

---

## 3. Sparse Attention Patterns

**Problem:** O(n²) is too expensive for long sequences.

**Solution:** Only compute attention for a subset of positions.

### 3a. Sliding Window Attention (Longformer, 2020)

**Used in:** Longformer, LED, BigBird

Each token only attends to **w neighbors** on each side:

```
Window size = 3

Token 5 attends to: [2, 3, 4, 5, 6, 7, 8]
                         ↑ current ↑
                    w=3 left    w=3 right
```

**Complexity:** O(n × w) — linear in sequence length!

### 3b. Dilated Sliding Window

**Used in:** Longformer (higher layers)

Like sliding window but with gaps (dilation):

```
Dilation = 2, Window = 3

Token 6 attends to: [0, 2, 4, 6, 8, 10, 12]
                              ↑
                         (every 2nd token)
```

Captures **longer-range dependencies** with same cost.

### 3c. Global + Local Attention (BigBird, 2020)

**Used in:** BigBird, Longformer

Some tokens (e.g., [CLS], special tokens) attend to **all positions**:

```
┌─────────────────────────────────────┐
│ ■ ■ ■ ■ ■ ■ ■ ■ ■ ■ ■ ■  ← [CLS] sees all (global)
│ ■ ■ ■ □ □ □ □ □ □ □ □ □  ← Token 1 (local window)
│ ■ ■ ■ ■ □ □ □ □ □ □ □ □  ← Token 2 (local window)
│ ■ □ ■ ■ ■ □ □ □ □ □ □ □  ← Token 3 (local + global)
│ ...
└─────────────────────────────────────┘
■ = attends, □ = masked
```

---

## 4. Linear Attention Variants

**Problem:** Softmax prevents efficient computation.

### 4a. Performer (2020)

**Used in:** Performer

Approximates softmax with random features:

```python
# Standard: softmax(QK^T) @ V  →  O(n²)
# Performer: φ(Q) @ (φ(K)^T @ V)  →  O(n)
```

Uses **FAVOR+** (Fast Attention Via positive Orthogonal Random features).

### 4b. Linear Transformer (2020)

Replace softmax with simple feature maps:

```python
# φ(x) = elu(x) + 1
attention = (φ(Q) @ φ(K)^T) @ V
```

---

## 5. Grouped Query Attention (GQA)

**Used in:** LLaMA 2, Mistral, Falcon

**Problem:** KV cache grows linearly with sequence length during inference.

**Solution:** Share K and V heads across multiple Q heads:

```
Standard MHA (8 heads):     GQA (8 Q, 2 KV groups):
Q: 8 heads                  Q: 8 heads
K: 8 heads                  K: 2 heads (shared by 4 Q each)  
V: 8 heads                  V: 2 heads (shared by 4 Q each)

Memory: 8 + 8 + 8 = 24      Memory: 8 + 2 + 2 = 12
```

### Variants

| Type | Q Heads | KV Heads | Memory |
|------|---------|----------|--------|
| **MHA** | n | n | High |
| **GQA** | n | n/k | Medium |
| **MQA** (Multi-Query) | n | 1 | Lowest |

---

## 6. Flash Attention (2022)

**Used in:** LLaMA 2, GPT-4, Claude, Mistral, most modern LLMs

**Not a new attention pattern** — same math, but **memory-efficient implementation**:

```
Standard:
1. Compute full QK^T matrix (n × n) → store in HBM
2. Apply softmax
3. Multiply by V

Flash Attention:
1. Tile Q, K, V into blocks
2. Compute attention block-by-block in SRAM (fast)
3. Never materialize full n × n matrix
```

### Benefits

| Metric | Standard | Flash Attention |
|--------|----------|-----------------|
| Memory | O(n²) | O(n) |
| Speed | 1x | 2-4x faster |
| Exact? | Yes | Yes (not approximate!) |

---

## 7. Rotary Position Embeddings (RoPE)

**Used in:** LLaMA, GPT-NeoX, PaLM, Mistral, most modern LLMs

Instead of adding position embeddings, **rotate** the query and key vectors:

```python
# Position m encodes as rotation in 2D subspaces
R_m = [[cos(mθ), -sin(mθ)],
       [sin(mθ),  cos(mθ)]]

q_rotated = R_m @ q
k_rotated = R_n @ k

# Attention score depends on relative position (m - n)
score = q_rotated · k_rotated = f(m - n)
```

### Why RoPE?

1. **Relative positions:** Score depends on distance, not absolute position
2. **Extrapolation:** Can extend to longer sequences than training
3. **Efficiency:** No extra parameters, just rotation

---

## 8. Sliding Window + Attention Sinks (StreamingLLM, 2023)

**Used in:** Streaming inference, very long contexts

**Discovery:** First few tokens ("attention sinks") receive disproportionate attention.

```
Keep: [0, 1, 2, 3] + [n-w, ..., n-1, n]
       ↑ sinks ↑     ↑ sliding window ↑
```

Enables **infinite context** during inference!

---

## 9. Mixture of Experts (MoE) with Attention

**Used in:** Mixtral, GPT-4 (rumored), Switch Transformer

Not attention itself, but often combined:

```
                    ┌─── Expert 1 ───┐
                    │                │
Input ──► Router ──►├─── Expert 2 ───┼──► Weighted Sum ──► Output
                    │                │
                    └─── Expert 8 ───┘
                    
Only top-k experts activated per token (e.g., k=2)
```

---

## Comparison Table: Attention in Popular Models

| Model | Attention Type | Context | KV Sharing | Position |
|-------|---------------|---------|------------|----------|
| **GPT-2** | Causal MHA | 1024 | None | Learned |
| **GPT-3** | Causal MHA | 2048 | None | Learned |
| **GPT-4** | Causal + MoE? | 8K-128K | Unknown | Unknown |
| **BERT** | Bidirectional MHA | 512 | None | Learned |
| **LLaMA 1** | Causal MHA | 2048 | None | RoPE |
| **LLaMA 2** | Causal GQA | 4096 | GQA | RoPE |
| **Mistral 7B** | Sliding Window GQA | 8K (32K effective) | GQA | RoPE |
| **Mixtral** | Sliding + MoE | 32K | GQA | RoPE |
| **Longformer** | Global + Local | 4096+ | None | Learned |
| **Claude** | Causal + Flash | 100K+ | Unknown | Unknown |

---

## Evolution Timeline

```
2017: Transformer (Vaswani)
      └─► Standard self-attention, O(n²)

2018: GPT-1, BERT
      └─► Causal vs Bidirectional masking

2019: Transformer-XL
      └─► Segment-level recurrence for longer context

2020: Longformer, BigBird, Performer
      └─► Sparse attention, O(n) complexity

2021: RoPE (Su et al.)
      └─► Rotary embeddings for relative positions

2022: Flash Attention (Dao et al.)
      └─► Memory-efficient exact attention

2023: LLaMA 2, Mistral
      └─► GQA + RoPE + Flash Attention

2023: StreamingLLM
      └─► Attention sinks for infinite context

2024: Ring Attention, Striped Attention
      └─► Distributed attention across devices
```

---

## Key Insights

1. **The core math is the same** — softmax(QK^T/√d) × V — but implementations vary wildly

2. **Memory is the bottleneck**, not compute — Flash Attention proves this

3. **Sparse patterns work** — you don't need full n² attention for good performance

4. **KV caching matters** — GQA/MQA reduce inference memory by 4-8x

5. **Position encoding is crucial** — RoPE enables length extrapolation

6. **Modern LLMs combine multiple techniques:**
   - Flash Attention (memory)
   - GQA (KV cache)
   - RoPE (positions)
   - Sliding window (long context)


In [47]:
x[0]

tensor([[ 0.1808, -0.0700],
        [-0.3596, -0.9152],
        [ 0.6258,  0.0255],
        [ 0.9545,  0.0643],
        [ 0.3612,  1.1679],
        [-1.3499, -0.5102],
        [ 0.2360, -0.2398],
        [-0.9211,  1.5433]])

In [48]:
xbow[0] # each row is the average of all the previous rows

tensor([[ 0.1808, -0.0700],
        [-0.0894, -0.4926],
        [ 0.1490, -0.3199],
        [ 0.3504, -0.2238],
        [ 0.3525,  0.0545],
        [ 0.0688, -0.0396],
        [ 0.0927, -0.0682],
        [-0.0341,  0.1332]])